# 02 - Resolve configuration from `dw.json`

The SDK reads the same `dw.json` file the B2C CLI uses. This example writes a
temp `dw.json` (with kebab-case keys and legacy aliases), loads it two ways, and
shows the normalized `NormalizedConfig` fields.

Public API: `load_dw_json`, `resolve_config`, `NormalizedConfig`,
`ResolveConfigOptions`.

In [ ]:
# --- Offline, credential-free setup -------------------------------------------
# Everything below runs with NO real network and NO real credentials. HTTP is
# mocked with respx, all state lives in a throwaway temp dir, and the auth-token
# caches are reset -- mirroring the SDK's own test harness (tests/conftest.py).
import base64
import json
import os
import tempfile
import time
from pathlib import Path

import httpx
import respx

from b2c_tooling_sdk.auth.oauth import reset_oauth_cache_for_testing
from b2c_tooling_sdk.auth.oauth_implicit import reset_implicit_cache_for_testing
from b2c_tooling_sdk.auth.oauth_pkce import reset_pkce_cache_for_testing
from b2c_tooling_sdk.auth.session_store import (
    FileAuthSessionBackend,
    set_auth_session_backend,
)

_tmp = Path(tempfile.mkdtemp(prefix="b2c-nb-"))
(_tmp / "data").mkdir(parents=True, exist_ok=True)
(_tmp / "config").mkdir(parents=True, exist_ok=True)

# Point every config/data dir at the temp dir so we never touch a real user store.
os.environ["XDG_DATA_HOME"] = str(_tmp / "data")
os.environ["XDG_CONFIG_HOME"] = str(_tmp / "config")
os.environ["LOCALAPPDATA"] = str(_tmp / "data")
os.environ.pop("B2C_CONFIG_DIR", None)

# Reset the module-level OAuth token caches for deterministic runs.
reset_oauth_cache_for_testing()
reset_pkce_cache_for_testing()
reset_implicit_cache_for_testing()

# Install a temp-dir file-backed auth-session store as the default.
set_auth_session_backend(FileAuthSessionBackend(_tmp / "store"))
print("Isolated temp dir:", _tmp)

## Write a temp `dw.json`

It mixes kebab-case keys (`code-version`, `client-id`) and **legacy aliases**
(`server` -> hostname, `secureHostname` -> webdavHostname, `scapi-shortcode`
-> shortCode).

In [ ]:
project_dir = _tmp / "project"
project_dir.mkdir(parents=True, exist_ok=True)
dw_json_path = project_dir / "dw.json"

dw_json_path.write_text(
    json.dumps(
        {
            "server": "example.demandware.net",          # legacy alias -> hostname
            "secureHostname": "dav.example.demandware.net",  # alias -> webdavHostname
            "code-version": "version1",
            "client-id": "aaaa-bbbb-cccc",
            "client-secret": "super-secret",
            "scapi-shortcode": "kv7kzm78",               # alias -> shortCode
            "tenant-id": "zzxy_prd",
        },
        indent=2,
    ),
    encoding="utf-8",
)
print(dw_json_path.read_text())

## `load_dw_json` -- raw load with camelCase normalization

`load_dw_json` reads a single file and normalizes every key (kebab/alias/camel)
to camelCase in a plain dict.

In [ ]:
from b2c_tooling_sdk import load_dw_json

loaded = load_dw_json(path=str(dw_json_path))
assert loaded is not None
print("hostname       :", loaded.config["hostname"])          # from `server`
print("webdavHostname :", loaded.config["webdavHostname"])    # from `secureHostname`
print("codeVersion    :", loaded.config["codeVersion"])
print("clientId       :", loaded.config["clientId"])
print("shortCode      :", loaded.config["shortCode"])         # from `scapi-shortcode`

## `resolve_config` -- the high-level resolver

`resolve_config` merges configuration sources and returns a resolved object whose
`.values` is a snake_case `NormalizedConfig`. Here we restrict resolution to a
single `DwJsonSource` rooted at our temp project dir for a deterministic result.

In [ ]:
from b2c_tooling_sdk import ResolveConfigOptions, resolve_config
from b2c_tooling_sdk.config import DwJsonSource

resolved = await resolve_config(
    options=ResolveConfigOptions(
        project_directory=str(project_dir),
        replace_default_sources=True,
        sources_before=[DwJsonSource()],
    )
)

cfg = resolved.values  # NormalizedConfig (snake_case fields)
print("hostname        :", cfg.hostname)
print("webdav_hostname :", cfg.webdav_hostname)
print("code_version    :", cfg.code_version)
print("client_id       :", cfg.client_id)
print("short_code      :", cfg.short_code)
print("tenant_id       :", cfg.tenant_id)
print("has_oauth_config:", resolved.has_oauth_config())

assert cfg.hostname == "example.demandware.net"
assert cfg.short_code == "kv7kzm78"

## Recap

- `load_dw_json` gives you the raw file contents with keys normalized to
  camelCase (aliases resolved).
- `resolve_config` returns a richer object; `.values` is a `NormalizedConfig`
  with snake_case fields ready for `create_instance_from_config` / `B2CInstance`.